In [1]:
import pandas as pd
import dill

In [2]:
import os

notebook_dir = r'/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks'
project_dir = os.path.dirname(notebook_dir)
print("Notebook Directory:", notebook_dir)
print("Project Directory:", project_dir)
os.chdir(project_dir)

Notebook Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks
Project Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Make sure to import ClassificationSpacyLemmaTokenizer from the source code.
from src.services.models.aftercourt_tokenizer import ClassificationSpacyLemmaTokenizer

In [4]:
VEC_PATH = "/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/models/vectorizers/05-12-2026_tf_idf_vectorizer_v1.dill"
DATA_PATH = "/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/data/PFUB_ERLASS_dataset_processed_with_geschaeftsnummer.csv"
MODEL_PATH = "/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/models/classification/12-01-2026_binary_pfub_rf_classifier_v1.dill"

In [5]:
tf_idf_vectorizer = dill.load(open(VEC_PATH, 'rb'))
data = pd.read_csv(DATA_PATH)
model = dill.load(open(MODEL_PATH, 'rb'))

In [6]:
data['is_pfub'].value_counts()

is_pfub
False    4563
True      468
Name: count, dtype: int64

In [7]:

X_text = data['text_w_tags']
y = data['is_pfub']

# train test split
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set size: {len(X_train_text)}, Test set size: {len(X_test_text)}")
print(f"Train set is pfub dist: {y_train.value_counts(normalize=True)}")
print(f"Test set is pfub dist: {y_test.value_counts(normalize=True)}")


Train set size: 4024, Test set size: 1007
Train set is pfub dist: is_pfub
False    0.907058
True     0.092942
Name: proportion, dtype: float64
Test set is pfub dist: is_pfub
False    0.906653
True     0.093347
Name: proportion, dtype: float64


In [8]:
X_train_tfidf = tf_idf_vectorizer.transform(X_train_text)
X_test_tfidf = tf_idf_vectorizer.transform(X_test_text)
# basic RF

from sklearn.ensemble import RandomForestClassifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_tfidf, y_train)

y_pred_probs = rf_classifier.predict_proba(X_test_tfidf)[:, 1]
document_types_test = data.loc[X_test_text.index, 'document_type']
doc_type_to_pred = pd.DataFrame({
    'document_type': document_types_test,
    'predicted_probs': y_pred_probs
})



In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots with 1 row and 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribution of Predicted Probabilities by Document Type (Box Plot)',
                    'Distribution of Predicted Probabilities by Document Type (Violin Plot)')
)

# Get unique document types
doc_types = doc_type_to_pred['document_type'].unique()

# Add box plots
for doc_type in doc_types:
    data_subset = doc_type_to_pred[doc_type_to_pred['document_type'] == doc_type]
    fig.add_trace(
        go.Box(
            y=data_subset['predicted_probs'],
            name=doc_type,
            boxmean='sd'
        ),
        row=1, col=1
    )

# Add violin plots
for doc_type in doc_types:
    data_subset = doc_type_to_pred[doc_type_to_pred['document_type'] == doc_type]
    fig.add_trace(
        go.Violin(
            y=data_subset['predicted_probs'],
            name=doc_type,
            box_visible=True,
            meanline_visible=True
        ),
        row=1, col=2
    )

# Add threshold line to violin plot
fig.add_hline(
    y=0.5, 
    line_dash="dash", 
    line_color="red",
    annotation_text="Threshold (0.5)",
    annotation_position="right",
    col=2
)

# Update layout
fig.update_layout(
    height=500,
    width=1400,
    showlegend=True,
    title_text="Predicted Probabilities Analysis by Document Type"
)

# Update y-axes labels
fig.update_yaxes(title_text="Predicted Probability", row=1, col=1)
fig.update_yaxes(title_text="Predicted Probability", row=1, col=2)

fig.show()

# Summary statistics by document type
print("\nSummary Statistics by Document Type:")
print(doc_type_to_pred.groupby('document_type')['predicted_probs'].describe())


Summary Statistics by Document Type:
                                        count      mean       std    min  \
document_type                                                              
approved_attachment_and_transfer_order   13.0  0.965017  0.082360  0.698   
approved_seizure                         81.0  0.961508  0.069441  0.680   
attachment_and_transfer_order           113.0  0.005543  0.043628  0.000   
contradiction                             2.0  0.203571  0.273751  0.010   
court_inbox                              82.0  0.017866  0.104566  0.000   
enforcement_order                         2.0  0.000000  0.000000  0.000   
ladung_va                               427.0  0.003379  0.013061  0.000   
mail_attachments                        181.0  0.008967  0.036600  0.000   
monierung_mb                            106.0  0.000660  0.005027  0.000   

                                             25%       50%       75%       max  
document_type                               